## Tutorial: Using AR (Adaptive Resampling) with a Custom Model

This notebook is a self-contained tutorial that shows **step by step** how to plug the AR module into your own example model.

#### What AR does

Standard mini-batch training draws cells uniformly at random, so rare cell types appear proportionally less often and can be under-learned. **AR (Adaptive Resampling)** changes sample combination without requiring cell-type labels:

1. After (or before) each epoch the encoder maps every training cell to a latent code *z*.
2. A per-dimension histogram estimates the density of *z*.
3. Cells in **low-density** latent regions (rare types) receive **higher sampling weights**.
4. Those weights drive mini-batch sampling for the next epoch.

#### What you need from your model

AR only requires that your model exposes a **latent representation** at the end of each epoch. Concretely:

```python
reconstruction, z = model(x)   # z is the latent code used by AR
```

If your model does not currently return `z`, you can add a small wrapper (shown in Section 3).

---
**Sections**
1. Imports
2. Load and preprocess data
3. Define your custom model
4. Hyperparameters
5. AR training loop — the core integration
6. Train and evaluate


### Imports

In [1]:
import sys, os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import scanpy as sc
import scvi
from sklearn.model_selection import train_test_split

# Make the AR module importable — assumes this notebook runs from the repo root.
# If you copy this notebook elsewhere, adjust the path to point at the repo.
sys.path.insert(0, os.path.abspath('.'))
from scripts.AR_resampling_weight_calculator import calculate_resampling_weights

### Load and preprocess data

We use the PBMC combined dataset (~11,990 cells, 9 cell types) from `scvi-tools` as example, which merges the 10x PBMC 8k and 4k datasets.

In [2]:
# Downloads ~50 MB on first run; subsequent runs use the local cache
adata = scvi.data.pbmc_dataset()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var.highly_variable].copy()

print(f"Preprocessed data shape: {adata.shape}")
print("\nCell type distribution:")
print(adata.obs["str_labels"].value_counts().to_string())

INFO     File data/gene_info_pbmc.csv already downloaded                                                           
INFO     File data/pbmc_metadata.pickle already downloaded                                                         
INFO     File data/pbmc8k/filtered_gene_bc_matrices.tar.gz already downloaded                                      
INFO     Extracting tar file                                                                                       
INFO     Removing extracted data at data/pbmc8k/filtered_gene_bc_matrices                                          
INFO     File data/pbmc4k/filtered_gene_bc_matrices.tar.gz already downloaded                                      
INFO     Extracting tar file                                                                                       
INFO     Removing extracted data at data/pbmc4k/filtered_gene_bc_matrices                                          
Preprocessed data shape: (11990, 2000)

Cell type distribution:
str_labe

In [3]:
X = adata.X.toarray().astype("float32")
X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train)
X_val_t   = torch.tensor(X_val)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}")

Train: (9592, 2000)  |  Val: (2398, 2000)


### Define your custom model

The example below is a lightweight MLP autoencoder. You can replace it with any architecture you like — a VAE, a transformer encoder, or an existing `scvi-tools` model wrapper — as long as the `forward` method returns `(reconstruction, z)` where `z` is the latent representation.

### Key interface requirement

```python
reconstruction, z = model(x)
```

- `x`  — input gene expression tensor of shape `(batch, n_genes)`
- `reconstruction` — model output of shape `(batch, n_genes)` used to compute the loss
- `z`  — latent code of shape `(batch, latent_dim)` used by AR to compute resampling weights

In [4]:
class CustomAutoencoder(nn.Module):
    """Example custom model: a three-layer MLP autoencoder.

    Replace this class with your own architecture. The only contract AR
    imposes is that forward() returns (reconstruction, z).
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=128, dropout_rate=0.2):
        super().__init__()

        # ── Encoder: maps gene expression → latent code z ─────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, latent_dim),
        )

        # ── Decoder: reconstructs gene expression from z ───────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 256),        nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, input_dim),         nn.Softplus(),  # non-negative output
        )

    def forward(self, x):
        z    = self.encoder(x)     # latent representation — used by AR
        recon = self.decoder(z)    # reconstruction — used for the loss
        return recon, z            # AR expects exactly this signature

### Hyperparameters

Standard training hyperparameters are grouped separately from the two AR-specific parameters (`BINS` and `SMOOTHING`) so it is easy to see what AR adds.

In [5]:
# ── Standard hyperparameters ────────────────────────────────────────────────
SEED         = 42
LATENT_DIM   = 32
BATCH_SIZE   = 512
NUM_EPOCHS   = 100
LR           = 1e-3
DROPOUT      = 0.2
WEIGHT_DECAY = 1e-4

# ── AR-specific hyperparameters ─────────────────────────────────────────────
# BINS      controls how finely the latent space is discretised for density
#           estimation. Default: 100.
BINS      = 100

# SMOOTHING prevents division by zero in empty histogram bins. 
SMOOTHING = 0.001

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


### AR training loop — the core integration

In [7]:
def train_with_AR(
    model,
    X_train_t,
    X_val_t,
    num_epochs,
    batch_size,
    lr,
    weight_decay,
    device,
    bins=100,
    smoothing_fac=0.001,
    loss_fn=None,
):
    """
    Train `model` with Adaptive Resampling (AR).

    Parameters
    ----------
    model        : nn.Module with signature forward(x) -> (recon, z)
    X_train_t    : torch.Tensor, shape (n_train, n_features)
    X_val_t      : torch.Tensor, shape (n_val,   n_features)
    num_epochs   : int
    batch_size   : int
    lr           : float  — learning rate for Adam
    weight_decay : float  — L2 regularisation coefficient
    device       : torch.device
    bins         : int    — histogram bins for AR density estimation
    smoothing_fac: float  — prevents zero-density bins
    loss_fn      : callable or None — defaults to nn.MSELoss()

    Returns
    -------
    train_losses, val_losses : lists of per-epoch average losses
    """
    if loss_fn is None:
        loss_fn = nn.MSELoss()

    optimizer      = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_train        = len(X_train_t)
    steps_per_epoch = n_train // batch_size

    train_losses, val_losses = [], []
    w = None  # epoch 0 uses uniform sampling (weights=None in random.choices)

    for epoch in range(num_epochs):

        # ── AR Step 1: encode the full training set to obtain latent codes ──
        # Skip on the very first epoch because the model has not yet learned
        # a meaningful latent space — uniform sampling is used instead.
        if epoch > 0:
            model.eval()
            with torch.no_grad():
                _, z = model(X_train_t.to(device))

            # compute per-cell resampling weights
            w = calculate_resampling_weights(
                z.cpu().numpy(),
                bins=bins,
                smoothing_fac=smoothing_fac,
            )

        model.train()
        running_loss = 0.0

        for _ in range(steps_per_epoch):

            # weighted sampling
            idx   = random.choices(range(n_train), weights=w, k=batch_size)
            batch = X_train_t[idx].to(device)

            optimizer.zero_grad()
            recon, _ = model(batch)     # latent z not needed during the update step
            loss = loss_fn(recon, batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_size

        train_losses.append(running_loss / (steps_per_epoch * batch_size))

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            recon_val, _ = model(X_val_t.to(device))
            val_loss = loss_fn(recon_val, X_val_t.to(device)).item()
        val_losses.append(val_loss)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1:3d}/{num_epochs}]  train={train_losses[-1]:.4f}  val={val_loss:.4f}")

    return train_losses, val_losses

### Train

In [8]:
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

model = CustomAutoencoder(
    input_dim=X_train.shape[1],
    latent_dim=LATENT_DIM,
    dropout_rate=DROPOUT,
).to(device)

print("Training with AR")
print("-" * 50)

train_losses, val_losses = train_with_AR(
    model,
    X_train_t,
    X_val_t,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    device=device,
    bins=BINS,
    smoothing_fac=SMOOTHING,
)

print(f"\nFinal training loss : {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")

Training with AR
--------------------------------------------------
  Epoch [ 10/100]  train=0.3784  val=0.3586
  Epoch [ 20/100]  train=0.3622  val=0.3513
  Epoch [ 30/100]  train=0.3594  val=0.3510
  Epoch [ 40/100]  train=0.3609  val=0.3497
  Epoch [ 50/100]  train=0.3600  val=0.3498
  Epoch [ 60/100]  train=0.3591  val=0.3500
  Epoch [ 70/100]  train=0.3563  val=0.3491
  Epoch [ 80/100]  train=0.3564  val=0.3496
  Epoch [ 90/100]  train=0.3535  val=0.3491
  Epoch [100/100]  train=0.3534  val=0.3488

Final training loss : 0.3534
Final validation loss: 0.3488
